In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import tempfile

# 비어 있는 경로에 Google Drive 연결
MOUNT_ROOT = Path(tempfile.mkdtemp(prefix="maek_drive_", dir="/content"))
drive.mount(str(MOUNT_ROOT))

DRIVE_ROOT = MOUNT_ROOT / "MyDrive" / "MAEK"
assert DRIVE_ROOT.is_dir(), "연결한 계정의 내 드라이브에 MAEK 폴더가 없습니다."

# MAEK 안에 UT 폴더 생성
UT_DRIVE = DRIVE_ROOT / "UT"
UT_DRIVE.mkdir(exist_ok=True)

# 실험별로 폴더를 나눠 기존 결과 보존
RUN_ID = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
DRIVE_RUN = UT_DRIVE / RUN_ID
DRIVE_RUN.mkdir(exist_ok=False)

# 데이터 생성·학습은 Colab 로컬에서 수행
LOCAL_RUN = Path("/content/ut_cnn_runs") / RUN_ID
LOCAL_RUN.mkdir(parents=True, exist_ok=False)
BASELINE = LOCAL_RUN / "baseline"

print("Drive 저장 위치:", DRIVE_RUN)
print("로컬 작업 위치:", LOCAL_RUN)

Mounted at /content/maek_drive_9gj80_mv
Drive 저장 위치: /content/maek_drive_9gj80_mv/MyDrive/MAEK/UT/20261109_190635
로컬 작업 위치: /content/ut_cnn_runs/20261109_190635


In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), "GPU 런타임으로 변경한 뒤 다시 실행하세요."

torch.set_num_threads(1)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

REPO = LOCAL_RUN / "repo"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/nalziori/MAEK.git", str(REPO)],
        check=True,
    )

assert (REPO / ".git").is_dir(), f"저장소 상태를 확인하세요: {REPO}"

COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

commit_record = DRIVE_RUN / "code_commit.txt"
if commit_record.exists():
    assert commit_record.read_text().strip() == COMMIT, "기록된 코드 버전과 다릅니다."
else:
    commit_record.write_text(COMMIT + "\n", encoding="utf-8")

print("코드 버전:", COMMIT)

GPU: Tesla T4
PyTorch: 2.14.1+cu130
CUDA: 13.0
코드 버전: f6fa8e95847bb15ad50a0c156cf9649a9a91dffa


In [ ]:
import sys
import json
import platform
import importlib.metadata as metadata

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(REPO / "requirements.txt"),
        "scipy",
    ],
    check=True,
)

environment = {
    "python": platform.python_version(),
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "commit": COMMIT,
    "packages": {
        name: metadata.version(name)
        for name in ("numpy", "pandas", "pyyaml", "pillow", "scipy")
    },
}

(DRIVE_RUN / "environment.json").write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
(DRIVE_RUN / "requirements-freeze.txt").write_text(freeze, encoding="utf-8")

print(json.dumps(environment, indent=2, ensure_ascii=False))

{
  "python": "3.13.16",
  "gpu": "Tesla T4",
  "torch": "2.14.1+cu130",
  "cuda": "13.0",
  "commit": "f6fa8e95847bb15ad50a0c156cf9649a9a91dffa",
  "packages": {
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "pyyaml": "6.0.3",
    "pillow": "11.3.0",
    "scipy": "1.16.3"
  }
}


In [ ]:
%pip install -q -r /content/MAEK/requirements.txt scipy

In [ ]:
import shutil
from google.colab import files

MODEL_DIR = BASELINE / "normal_ut_reviewed_v1"
SPEC_DIR = BASELINE / "normal_ut_repair_v2"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SPEC_DIR.mkdir(parents=True, exist_ok=True)

def copy_once(source, destination):
    source, destination = Path(source), Path(destination)
    assert source.is_file(), f"필요한 파일이 없습니다: {source}"

    if destination.exists():
        assert destination.read_bytes() == source.read_bytes(), (
            f"기존 파일과 내용이 다릅니다. 덮어쓰지 않았습니다: {destination}"
        )
    else:
        shutil.copy2(source, destination)

for name in (
    "mlp_1601.pt",
    "mlp_1602.pt",
    "mlp_1603.pt",
    "calibrated_config.json",
):
    copy_once(REPO / "UT" / "normal_ut_reviewed_v1" / name, MODEL_DIR / name)

copy_once(
    REPO / "UT" / "normal_ut_repair_v2" / "frozen_decisions.json",
    SPEC_DIR / "frozen_decisions.json",
)

protocol_path = SPEC_DIR / "protocol.json"

if not protocol_path.is_file():
    candidates = [
        UT_DRIVE / "protocol.json",
        DRIVE_ROOT / "protocol.json",
        REPO / "UT" / "normal_ut_repair_v2" / "protocol.json",
        DRIVE_ROOT / "초음파피팅" / "protocol.json",
    ]

    protocol_source = next((p for p in candidates if p.is_file()), None)

    if protocol_source is not None:
        copy_once(protocol_source, protocol_path)
        print("실험 명세 출처:", protocol_source)
    else:
        print("PC에서 protocol.json을 선택하세요.")
        uploaded = files.upload()
        assert "protocol.json" in uploaded, "protocol.json 파일을 선택해야 합니다."
        protocol_path.write_bytes(uploaded["protocol.json"])

print("기존 모델:", MODEL_DIR)
print("실험 명세:", protocol_path)

실험 명세 출처: /content/maek_drive_1uof8fob/MyDrive/MAEK/protocol.json
기존 모델: /content/ut_cnn_runs/20261109_162703/baseline/normal_ut_reviewed_v1
실험 명세: /content/ut_cnn_runs/20261109_162703/baseline/normal_ut_repair_v2/protocol.json


In [ ]:
import hashlib

frozen = json.loads(
    (SPEC_DIR / "frozen_decisions.json").read_text(encoding="utf-8")
)

def expected_hash(filename):
    matches = [
        digest
        for recorded_path, digest in frozen["hashes"].items()
        if recorded_path.replace("\\", "/").rsplit("/", 1)[-1] == filename
    ]
    assert len(matches) == 1, f"해시 기록이 없거나 중복됩니다: {filename}"
    return matches[0]

verified = {}

for path in [
    MODEL_DIR / "mlp_1601.pt",
    MODEL_DIR / "mlp_1602.pt",
    MODEL_DIR / "mlp_1603.pt",
    MODEL_DIR / "calibrated_config.json",
    protocol_path,
]:
    raw = path.read_bytes()
    raw_hash = hashlib.sha256(raw).hexdigest()
    accepted_hashes = {raw_hash}

    # JSON은 Windows/Linux 줄바꿈 차이만 허용
    if path.suffix == ".json":
        accepted_hashes.add(
            hashlib.sha256(raw.replace(b"\r\n", b"\n")).hexdigest()
        )
        json.loads(raw.decode("utf-8"))

    assert expected_hash(path.name) in accepted_hashes, (
        f"원본 기록과 파일이 다릅니다: {path.name}"
    )

    verified[str(path.relative_to(BASELINE))] = raw_hash
    print("검증 통과:", path.name)

# 체크포인트를 실제로 읽을 수 있는지도 확인
for seed in (1601, 1602, 1603):
    checkpoint = torch.load(
        MODEL_DIR / f"mlp_{seed}.pt",
        map_location="cpu",
        weights_only=True,
    )
    assert {"state", "mean", "std"}.issubset(checkpoint), (
        f"체크포인트 구조 확인 필요: mlp_{seed}.pt"
    )

# 검증된 파일을 Drive에 보관. 기존 파일이 다르면 중단.
for source in sorted(BASELINE.rglob("*")):
    if source.is_file():
        destination = DRIVE_RUN / "baseline" / source.relative_to(BASELINE)
        destination.parent.mkdir(parents=True, exist_ok=True)
        copy_once(source, destination)
        assert destination.read_bytes() == source.read_bytes()

(DRIVE_RUN / "baseline_sha256.json").write_text(
    json.dumps(verified, indent=2),
    encoding="utf-8",
)

print("\n준비 완료")
print("저장 위치:", DRIVE_RUN)
print("다음 단계: CNN 실험 조건 설정 → 파형 생성 → 학습")

검증 통과: mlp_1601.pt
검증 통과: mlp_1602.pt
검증 통과: mlp_1603.pt
검증 통과: calibrated_config.json
검증 통과: protocol.json

준비 완료
저장 위치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703
다음 단계: CNN 실험 조건 설정 → 파형 생성 → 학습


In [ ]:
import hashlib
import shutil
import zipfile
import io
import numpy as np
import pandas as pd

candidates = list(DRIVE_ROOT.glob("*/maek-ut.zip"))
assert len(candidates) == 1, (
    f"maek-ut.zip 경로를 확인하세요: {candidates}"
)

source = candidates[0]
DATA_DIR = LOCAL_RUN / "existing_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / "maek-ut.zip"

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not archive.exists():
    shutil.copy2(source, archive)

assert file_sha256(source) == file_sha256(archive), "복사본 해시 불일치"

def load_straight(z, split):
    with np.load(
        io.BytesIO(z.read(f"ut/{split}/ut_straight.npz")),
        allow_pickle=False,
    ) as arrays:
        signals = arrays["signals"].astype(np.float32)

    labels = pd.read_csv(z.open(f"ut/{split}/ut_labels.csv"))
    target = (labels["pit_depth_mm"].to_numpy() > 0).astype(np.int64)

    assert signals.shape == (len(labels), 4096)
    assert np.isfinite(signals).all()
    assert set(np.unique(target)) == {0, 1}

    return signals, labels, target

with zipfile.ZipFile(archive) as z:
    X_existing, metadata_existing, y_existing = load_straight(z, "train")

    # shifted는 파일 존재만 확인하고, 모델 확정 후 평가할 때 로드
    assert "ut/shifted/ut_straight.npz" in z.namelist()
    assert "ut/shifted/ut_labels.csv" in z.namelist()

manifest = {
    "source": str(source),
    "sha256": file_sha256(archive),
    "dataset": "data-v3",
    "input": "ut_straight.npz/signals",
    "target": "pit_depth_mm > 0",
    "shape": list(X_existing.shape),
    "normal_count": int((y_existing == 0).sum()),
    "pit_count": int((y_existing == 1).sum()),
    "n_avg": sorted(
        int(v) for v in metadata_existing["n_avg"].unique()
    ),
}

(DRIVE_RUN / "existing_data_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("shifted 데이터는 최종 평가까지 사용하지 않습니다.")

{
  "source": "/content/maek_drive_1uof8fob/MyDrive/MAEK/data-v3 (합성 데이터 최신)/maek-ut.zip",
  "sha256": "70272e70b16c216b7a4b84f6d747de12eb2b8d5989c788190e423f181e4ccd6b",
  "dataset": "data-v3",
  "input": "ut_straight.npz/signals",
  "target": "pit_depth_mm > 0",
  "shape": [
    10000,
    4096
  ],
  "normal_count": 7021,
  "pit_count": 2979,
  "n_avg": [
    16
  ]
}
shifted 데이터는 최종 평가까지 사용하지 않습니다.


In [ ]:
import random
import hashlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

SEED = 202611091
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# 완전히 같은 파형이 여러 분할에 들어가는 것을 방지.
# 이 검사는 동일 형상의 다른 측정까지 식별하지는 못합니다.
waveform_hashes = [
    hashlib.sha256(row.tobytes()).digest()
    for row in X_existing
]
assert len(set(waveform_hashes)) == len(X_existing), (
    "중복 파형이 있습니다. 그룹 단위 분리가 필요합니다."
)

rng = np.random.default_rng(SEED)
split_indices = {
    name: [] for name in ("train", "validation", "calibration", "test")
}

# 각 분할의 정상/피팅 비율 유지
for label in (0, 1):
    indices = np.flatnonzero(y_existing == label)
    rng.shuffle(indices)
    boundaries = [
        int(len(indices) * 0.60),
        int(len(indices) * 0.75),
        int(len(indices) * 0.90),
    ]

    for name, part in zip(
        split_indices, np.split(indices, boundaries)
    ):
        split_indices[name].extend(part.tolist())

for name in split_indices:
    split_indices[name] = np.array(split_indices[name], dtype=np.int64)
    rng.shuffle(split_indices[name])

all_indices = np.concatenate(list(split_indices.values()))
assert len(all_indices) == len(X_existing)
assert len(np.unique(all_indices)) == len(X_existing)

train_idx = split_indices["train"]
RF_MEAN = float(X_existing[train_idx].mean(dtype=np.float64))
RF_STD = float(X_existing[train_idx].std(dtype=np.float64))
assert RF_STD > 0

# 파형별 최대값 정규화 대신 공통 스케일 사용: 진폭 차이 보존
def make_dataset(indices):
    x = (X_existing[indices] - RF_MEAN) / RF_STD
    y = y_existing[indices].astype(np.float32)
    return TensorDataset(
        torch.from_numpy(x[:, None, :]),
        torch.from_numpy(y),
    )

train_loader = DataLoader(
    make_dataset(train_idx),
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED),
)

val_loader = DataLoader(
    make_dataset(split_indices["validation"]),
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# 재실행 시 동일 분할인지 확인
split_path = DRIVE_RUN / "cnn_split_indices.npz"
if split_path.exists():
    with np.load(split_path, allow_pickle=False) as saved:
        for name, indices in split_indices.items():
            assert np.array_equal(saved[name], indices)
else:
    np.savez_compressed(split_path, **split_indices)

preprocessing = {
    "seed": SEED,
    "rf_mean": RF_MEAN,
    "rf_std": RF_STD,
    "input": "RF only, shape=(batch, 1, 4096)",
    "target": "pit_depth_mm > 0",
    "dataset_sha256": manifest["sha256"],
}
(DRIVE_RUN / "cnn_preprocessing.json").write_text(
    json.dumps(preprocessing, indent=2),
    encoding="utf-8",
)

for name, indices in split_indices.items():
    print(
        f"{name:12s}: {len(indices):5d}개 / "
        f"정상 {(y_existing[indices] == 0).sum():4d} / "
        f"피팅 {(y_existing[indices] == 1).sum():4d}"
    )

print("학습 데이터 정규화:", RF_MEAN, RF_STD)

train       :  5999개 / 정상 4212 / 피팅 1787
validation  :  1500개 / 정상 1053 / 피팅  447
calibration :  1500개 / 정상 1053 / 피팅  447
test        :  1001개 / 정상  703 / 피팅  298
학습 데이터 정규화: 0.0007120994117721202 0.07099489691488567


In [ ]:
import importlib.util

MODEL_SOURCE = '''
import torch
from torch import nn

class PitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.AdaptiveAvgPool1d(32),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(-1)
'''

model_path = LOCAL_RUN / "pit_cnn.py"
model_path.write_text(MODEL_SOURCE, encoding="utf-8")
copy_once(model_path, DRIVE_RUN / "pit_cnn.py")

spec = importlib.util.spec_from_file_location("pit_cnn", model_path)
cnn_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cnn_module)
PitCNN = cnn_module.PitCNN

device = torch.device("cuda")
assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."

model = PitCNN().to(device)

with torch.inference_mode():
    output = model(torch.zeros(2, 1, 4096, device=device))
assert output.shape == (2,)
assert torch.isfinite(output).all()

print(model)
print("학습 파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")
print("입출력 확인: (2, 1, 4096) →", tuple(output.shape))

PitCNN(
  (features): Sequential(
    (0): Conv1d(1, 16, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): AdaptiveAvgPool1d(output_size=32)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=64, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)
학습 파라미터: 145,281
입출력 확인: (2, 1, 4096) → (2,)


In [ ]:
import time
import math
from torch import nn

BEST_LOCAL = LOCAL_RUN / "cnn_best.pt"
BEST_DRIVE = DRIVE_RUN / "cnn_best.pt"

assert not BEST_DRIVE.exists(), (
    "이 실행 폴더에 학습된 CNN이 있습니다. "
    "덮어쓰지 않도록 중단했습니다."
)

# 앞 셀을 반복 실행했어도 초기 가중치를 동일하게 시작
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
model = PitCNN().to(device)

train_labels = y_existing[train_idx]
pos_weight = float(
    (train_labels == 0).sum() / (train_labels == 1).sum()
)

train_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=device)
)
val_loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

MAX_EPOCHS = 40
PATIENCE = 8
best_loss = float("inf")
bad_epochs = 0
history = []
started = time.perf_counter()

training_config = {
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "batch_size": 128,
    "optimizer": "AdamW",
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "pos_weight": pos_weight,
    "selection": "minimum validation BCE",
    "target_far": 0.007,
}
(DRIVE_RUN / "cnn_training_config.json").write_text(
    json.dumps(training_config, indent=2),
    encoding="utf-8",
)

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()
    train_sum = 0.0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = train_loss_fn(logits, y)
        assert torch.isfinite(loss), "학습 손실이 NaN/Inf입니다."

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_sum += loss.item() * len(y)

    model.eval()
    val_sum = 0.0

    with torch.inference_mode():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            loss = val_loss_fn(model(x), y)
            val_sum += loss.item() * len(y)

    train_loss = train_sum / len(train_loader.dataset)
    val_loss = val_sum / len(val_loader.dataset)
    assert math.isfinite(val_loss), "검증 손실이 NaN/Inf입니다."

    improved = val_loss < best_loss

    if improved:
        best_loss = val_loss
        bad_epochs = 0

        torch.save(
            {
                "state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "epoch": epoch,
                "validation_loss": val_loss,
                "preprocessing": preprocessing,
                "training_config": training_config,
                "code_commit": COMMIT,
                "model_source_sha256": file_sha256(model_path),
            },
            BEST_LOCAL,
        )
        # この実行でのベストモデルを更新
        shutil.copy2(BEST_LOCAL, BEST_DRIVE)
    else:
        bad_epochs += 1

    row = {
        "epoch": epoch,
        "train_weighted_bce": train_loss,
        "validation_bce": val_loss,
        "seconds": time.perf_counter() - epoch_start,
        "best": improved,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(
        DRIVE_RUN / "cnn_training_history.csv", index=False
    )

    print(
        f"{epoch:02d}/{MAX_EPOCHS} "
        f"train={train_loss:.5f} "
        f"val={val_loss:.5f} "
        f"{row['seconds']:.1f}s "
        f"{'★ best' if improved else ''}",
        flush=True,
    )

    if bad_epochs >= PATIENCE:
        print("검증 손실 개선이 없어 조기 종료합니다.")
        break

checkpoint = torch.load(
    BEST_LOCAL, map_location="cpu", weights_only=True
)
model.load_state_dict(checkpoint["state_dict"])
model.to(device).eval()

assert file_sha256(BEST_LOCAL) == file_sha256(BEST_DRIVE)

elapsed = time.perf_counter() - started
print(f"\n학습 완료: {elapsed / 60:.1f}분")
print("선택 epoch:", checkpoint["epoch"])
print("최소 검증 BCE:", checkpoint["validation_loss"])
print("저장:", BEST_DRIVE)
print("다음 단계: 정상 보정 데이터로 임계값 확정 → CNN/MLP 평가 비교")

01/40 train=0.95576 val=0.62015 1.6s ★ best
02/40 train=0.57433 val=0.10582 1.0s ★ best
03/40 train=0.09606 val=0.02639 1.0s ★ best
04/40 train=0.05007 val=0.01156 1.0s ★ best
05/40 train=0.02930 val=0.01827 0.9s 
06/40 train=0.01766 val=0.00974 1.0s ★ best
07/40 train=0.01685 val=0.00665 1.0s ★ best
08/40 train=0.01775 val=0.00692 1.0s 
09/40 train=0.01603 val=0.00769 1.0s 
10/40 train=0.01305 val=0.02019 1.0s 
11/40 train=0.02695 val=0.00648 1.0s ★ best
12/40 train=0.02293 val=0.00617 1.0s ★ best
13/40 train=0.01408 val=0.00315 1.0s ★ best
14/40 train=0.00752 val=0.01111 1.0s 
15/40 train=0.01353 val=0.00761 1.0s 
16/40 train=0.01354 val=0.00200 1.0s ★ best
17/40 train=0.01471 val=0.00325 1.0s 
18/40 train=0.02038 val=0.00345 1.0s 
19/40 train=0.02477 val=0.00791 1.0s 
20/40 train=0.01565 val=0.00191 1.0s ★ best
21/40 train=0.00912 val=0.00313 1.0s 
22/40 train=0.00374 val=0.00208 1.0s 
23/40 train=0.00376 val=0.00155 1.0s ★ best
24/40 train=0.00466 val=0.00245 1.0s 
25/40 train=0.00

In [ ]:
import json
import hashlib
import numpy as np
import torch
from scipy.stats import beta

TARGET_FAR = 0.007
CONFIDENCE = 0.95

# 10번 셀에서 저장한 최적 모델 불러오기
checkpoint_path = DRIVE_RUN / "cnn_best.pt"
assert checkpoint_path.is_file(), "10번 셀의 학습을 먼저 완료하세요."

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

model = PitCNN().to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# 학습 당시 저장한 정규화 값 사용
saved_preprocessing = checkpoint["preprocessing"]
rf_mean = float(saved_preprocessing["rf_mean"])
rf_std = float(saved_preprocessing["rf_std"])

assert rf_std > 0
assert saved_preprocessing["dataset_sha256"] == manifest["sha256"]

# 보정 데이터 중 정상 파형만 선택
calibration_idx = split_indices["calibration"]
normal_idx = calibration_idx[y_existing[calibration_idx] == 0]

assert len(normal_idx) > 0, "정상 보정 데이터가 없습니다."

# 학습·검증·일반 평가 데이터와 분리됐는지 재확인
for name in ("train", "validation", "test"):
    assert not np.intersect1d(
        calibration_idx, split_indices[name]
    ).size, f"{name} 데이터와 보정 데이터가 겹칩니다."

normal_scores = []

with torch.inference_mode():
    for start in range(0, len(normal_idx), 256):
        indices = normal_idx[start:start + 256]
        x = (X_existing[indices] - rf_mean) / rf_std
        x = torch.from_numpy(
            np.ascontiguousarray(x[:, None, :], dtype=np.float32)
        ).to(device)

        scores = torch.sigmoid(model(x))
        normal_scores.append(scores.cpu().numpy())

normal_scores = np.concatenate(normal_scores).astype(np.float64)
assert np.isfinite(normal_scores).all()

n_normal = len(normal_scores)

# 순서통계량을 이용한 보수적 임계값 선택.
# 정상 점수가 같은 분포에서 독립적으로 나온다는 가정 아래,
# FAR 상한이 목표 이하가 되는 허용 초과 개수를 선택합니다.
allowed_counts = np.arange(n_normal)
upper_bounds = beta.ppf(
    CONFIDENCE,
    allowed_counts + 1,
    n_normal - allowed_counts,
)

eligible = allowed_counts[upper_bounds <= TARGET_FAR]

assert len(eligible) > 0, (
    f"정상 보정 데이터 {n_normal}개로는 "
    f"{CONFIDENCE:.0%} 기준에서 목표 FAR {TARGET_FAR:.2%}를 "
    "설정하기에 부족합니다. 독립적인 정상 보정 데이터가 더 필요합니다."
)

allowed_fp = int(eligible[-1])
sorted_scores = np.sort(normal_scores)
threshold = float(sorted_scores[n_normal - allowed_fp - 1])

# 이후 평가에서도 반드시 '초과(>)'를 사용합니다.
observed_fp = int(np.sum(normal_scores > threshold))
observed_far = observed_fp / n_normal
far_upper = float(upper_bounds[allowed_fp])

assert observed_fp <= allowed_fp

threshold_record = {
    "model": "PitCNN",
    "selected_epoch": int(checkpoint["epoch"]),
    "model_sha256": file_sha256(checkpoint_path),
    "dataset_sha256": manifest["sha256"],
    "split_sha256": file_sha256(DRIVE_RUN / "cnn_split_indices.npz"),
    "target_far": TARGET_FAR,
    "confidence": CONFIDENCE,
    "threshold": threshold,
    "decision_rule": "score > threshold",
    "normal_calibration_count": n_normal,
    "allowed_exceedances": allowed_fp,
    "observed_calibration_fp": observed_fp,
    "observed_calibration_far": observed_far,
    "order_statistic_far_upper": far_upper,
    "scope": "정상 보정 데이터와 동일한 분포에 대한 기준",
}

threshold_path = DRIVE_RUN / "cnn_threshold.json"

# 이미 확정된 기준과 다르면 덮어쓰지 않음
if threshold_path.exists():
    previous = json.loads(threshold_path.read_text(encoding="utf-8"))
    assert previous == threshold_record, (
        "이미 저장된 판정 기준과 다릅니다. 기존 기준은 변경하지 않았습니다."
    )
else:
    threshold_path.write_text(
        json.dumps(threshold_record, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

CNN_THRESHOLD = threshold

print("CNN 판정 임계값 확정")
print(f"선택 모델: {checkpoint['epoch']} epoch")
print(f"정상 보정 데이터: {n_normal:,}개")
print(f"판정 조건: 피팅 점수 > {CNN_THRESHOLD:.10g}")
print(f"보정 데이터 오탐: {observed_fp}/{n_normal}")
print(f"보정 데이터 오탐률: {observed_far:.3%}")
print(f"동일 분포 가정의 FAR 상한: {far_upper:.3%}")
print(f"목표 FAR: {TARGET_FAR:.3%}")
print("저장 위치:", threshold_path)
print("\n다음 단계: 임계값을 변경하지 않고 일반·shifted 데이터 평가")

CNN 판정 임계값 확정
선택 모델: 27 epoch
정상 보정 데이터: 1,053개
판정 조건: 피팅 점수 > 0.004100258462
보정 데이터 오탐: 2/1053
보정 데이터 오탐률: 0.190%
동일 분포 가정의 FAR 상한: 0.597%
목표 FAR: 0.700%
저장 위치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_threshold.json

다음 단계: 임계값을 변경하지 않고 일반·shifted 데이터 평가


In [ ]:

import time
import zipfile
import numpy as np
import pandas as pd
import torch
from scipy.stats import beta

# 확정된 모델·데이터·분할인지 확인
assert file_sha256(checkpoint_path) == threshold_record["model_sha256"]
assert file_sha256(archive) == threshold_record["dataset_sha256"]
assert file_sha256(
    DRIVE_RUN / "cnn_split_indices.npz"
) == threshold_record["split_sha256"]

# 메모리에 있는 모델도 저장된 최적 가중치로 복원
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)
model.load_state_dict(checkpoint["state_dict"])
model.to(device).eval()

rf_mean = float(checkpoint["preprocessing"]["rf_mean"])
rf_std = float(checkpoint["preprocessing"]["rf_std"])
CNN_THRESHOLD = float(threshold_record["threshold"])

# 일반 평가: 학습 전에 분리해 둔 데이터
test_idx = split_indices["test"]
X_test = X_existing[test_idx]
y_test = y_existing[test_idx]

# shifted는 이 단계에서 처음 불러옴
with zipfile.ZipFile(archive) as z:
    X_shifted, metadata_shifted, y_shifted = load_straight(z, "shifted")

assert set(metadata_shifted["n_avg"].unique()) == {16}

# shifted에 기존 데이터와 완전히 같은 파형이 있는지 검사
existing_hashes = set(waveform_hashes)
assert all(
    hashlib.sha256(row.tobytes()).digest() not in existing_hashes
    for row in X_shifted
), "shifted 데이터에 기존 데이터와 동일한 파형이 있습니다."

def proportion_ci(count, total):
    """이항비율의 양측 95% Clopper–Pearson 신뢰구간."""
    assert total > 0
    lower = 0.0 if count == 0 else beta.ppf(
        0.025, count, total - count + 1
    )
    upper = 1.0 if count == total else beta.ppf(
        0.975, count + 1, total - count
    )
    return float(lower), float(upper)

def predict_cnn(signals, batch_size=256):
    # 정규화와 파일 읽기는 시간 측정에서 제외
    normalized = np.ascontiguousarray(
        ((signals - rf_mean) / rf_std)[:, None, :],
        dtype=np.float32,
    )
    outputs = []

    torch.cuda.synchronize()
    started = time.perf_counter()

    with torch.inference_mode():
        for start in range(0, len(normalized), batch_size):
            x = torch.from_numpy(
                normalized[start:start + batch_size]
            ).to(device)
            scores = torch.sigmoid(model(x))
            outputs.append(scores.cpu().numpy())

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    scores = np.concatenate(outputs)
    assert np.isfinite(scores).all()
    return scores, elapsed

# GPU 초기 실행 비용을 줄이기 위한 워밍업
with torch.inference_mode():
    for _ in range(3):
        model(torch.zeros(256, 1, 4096, device=device))
torch.cuda.synchronize()

results = []
prediction_frames = []

for name, signals, labels, row_ids in [
    ("ordinary", X_test, y_test, test_idx),
    ("shifted", X_shifted, y_shifted, np.arange(len(y_shifted))),
]:
    scores, seconds = predict_cnn(signals)
    predicted = scores > CNN_THRESHOLD
    positive = labels == 1
    negative = labels == 0

    tp = int(np.sum(predicted & positive))
    fn = int(np.sum(~predicted & positive))
    fp = int(np.sum(predicted & negative))
    tn = int(np.sum(~predicted & negative))

    n_positive = tp + fn
    n_negative = fp + tn
    recall = tp / n_positive
    far = fp / n_negative

    recall_low, recall_high = proportion_ci(tp, n_positive)
    far_low, far_high = proportion_ci(fp, n_negative)

    results.append({
        "model": "PitCNN",
        "dataset": name,
        "n": len(labels),
        "n_avg": 16,
        "threshold": CNN_THRESHOLD,
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
        "recall": recall,
        "recall_ci_low": recall_low,
        "recall_ci_high": recall_high,
        "far": far,
        "far_ci_low": far_low,
        "far_ci_high": far_high,
        "inference_seconds": seconds,
        "amortized_ms_per_waveform": seconds * 1000 / len(labels),
        "batch_size": 256,
        "device": torch.cuda.get_device_name(0),
    })

    prediction_frames.append(pd.DataFrame({
        "dataset": name,
        "source_row": row_ids,
        "label": labels,
        "score": scores,
        "prediction": predicted.astype(np.int64),
    }))

    print(f"\n[{name}] {len(labels):,}개")
    print(
        f"재현율: {recall:.2%} "
        f"(95% CI {recall_low:.2%}–{recall_high:.2%})"
    )
    print(
        f"오탐률: {far:.2%} "
        f"(95% CI {far_low:.2%}–{far_high:.2%})"
    )
    print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")
    print(f"추론 시간: {seconds:.3f}초")
    print(
        f"배치 처리 기준 파형당 평균: "
        f"{seconds * 1000 / len(labels):.4f}ms"
    )

cnn_results = pd.DataFrame(results)
cnn_predictions = pd.concat(prediction_frames, ignore_index=True)

cnn_results.to_csv(
    DRIVE_RUN / "cnn_evaluation.csv", index=False
)
cnn_predictions.to_csv(
    DRIVE_RUN / "cnn_predictions.csv", index=False
)

print("\n저장 완료:", DRIVE_RUN)
display(cnn_results)



[ordinary] 1,001개
재현율: 100.00% (95% CI 98.77%–100.00%)
오탐률: 0.28% (95% CI 0.03%–1.02%)
TP=298, FN=0, FP=2, TN=701
추론 시간: 0.057초
배치 처리 기준 파형당 평균: 0.0567ms

[shifted] 3,000개
재현율: 100.00% (95% CI 99.57%–100.00%)
오탐률: 88.72% (95% CI 87.30%–90.03%)
TP=864, FN=0, FP=1895, TN=241
추론 시간: 0.145초
배치 처리 기준 파형당 평균: 0.0482ms

저장 완료: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703


,model,dataset,n,n_avg,threshold,tp,fn,fp,tn,recall,recall_ci_low,recall_ci_high,far,far_ci_low,far_ci_high,inference_seconds,amortized_ms_per_waveform,batch_size,device
0,PitCNN,ordinary,1001,16,0.0041,298,0,2,701,1.0,0.987698,1.0,0.002845,0.000345,0.010239,0.056791,0.056734,256,Tesla T4
1,PitCNN,shifted,3000,16,0.0041,864,0,1895,241,1.0,0.995740,1.0,0.887172,0.872987,0.900285,0.144598,0.048199,256,Tesla T4


In [ ]:
import sys
import subprocess
import hashlib
import pandas as pd

FOLLOWUP = DRIVE_RUN / "cnn_followup_v2"
SCRIPT = FOLLOWUP / "cnn_followup.py"
EXPERIMENT = FOLLOWUP / "dr_experiment"

assert SCRIPT.is_file(), (
    "Drive 동기화 후 다시 실행하세요: "
    f"{SCRIPT}"
)

expected_sha = (
    "70f673cf23e4798100d8e661e33e825486"
    "56ae6dfd477c839fa947a710c604f4"
)
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == expected_sha

comparison = pd.read_csv(
    FOLLOWUP / "b_diagnosis" / "b_comparison.csv"
)
display(comparison[
    ["model", "domain", "recall", "far", "auc"]
])

def run_followup(stage):
    subprocess.run(
        [
            sys.executable, "-u", str(SCRIPT), stage,
            "--repo", str(REPO),
            "--run", str(DRIVE_RUN),
            "--archive", str(archive),
            "--out", str(EXPERIMENT),
        ],
        check=True,
    )


,model,domain,recall,far,auc
0,B_existing_MLP_no_hard_gate,ordinary,1.0000,0.0020,1.000000
1,C_MF_CFAR_no_extra_gate,ordinary,1.0000,0.0035,1.000000
2,D_new_MLP_no_hard_gate,ordinary,1.0000,0.0025,1.000000
3,CNN_v1_original,ordinary,1.0000,0.1880,0.999912
4,CNN_v1_recalibrated,ordinary,0.9400,0.0000,0.999912
5,B_existing_MLP_no_hard_gate,shifted,0.8825,0.0050,0.987820
6,C_MF_CFAR_no_extra_gate,shifted,0.8595,0.0035,0.974052
7,D_new_MLP_no_hard_gate,shifted,0.8755,0.0060,0.985624
8,CNN_v1_original,shifted,1.0000,0.9020,0.997971
9,CNN_v1_recalibrated,shifted,0.9695,0.0040,0.997971


In [ ]:
import subprocess
import sys
import hashlib

SCRIPT = FOLLOWUP / "cnn_followup_recovery.py"
PREVIOUS_EXPERIMENT = FOLLOWUP / "dr_experiment"
EXPERIMENT = FOLLOWUP / "dr_experiment_recovered"

assert SCRIPT.is_file(), "Drive 동기화 후 다시 실행하세요."
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == (
    "d3ba3df4af65f80a580195ccf0f50fa19"
    "d9feb9089c597397c2691cedfc7db74"
)

def run_followup(stage):
    command = [
        sys.executable, "-u", str(SCRIPT), stage,
        "--repo", str(REPO),
        "--run", str(DRIVE_RUN),
        "--archive", str(archive),
        "--out", str(EXPERIMENT),
    ]

    if stage == "prepare":
        command += ["--recover-from", str(PREVIOUS_EXPERIMENT)]

    print(f"[시작] {stage}", flush=True)
    print("저장 위치:", EXPERIMENT, flush=True)

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)

        return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(
                f"{stage} 실패: 종료 코드 {return_code}. 위 오류를 확인하세요."
            )

        print(f"\n[완료] {stage}", flush=True)

    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("\n프로세스를 중지했습니다.", flush=True)
        raise

    finally:
        process.stdout.close()

run_followup("prepare")

[시작] prepare
저장 위치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/dr_experiment_recovered
입력 파일 해시 확인 중
복구 검사: train
정상 파일 복구 완료: train
복구 검사: validation
정상 파일 복구 완료: validation
복구 검사: calibration_ordinary
정상 파일 복구 완료: calibration_ordinary
복구 검사: calibration_shifted
재생성 필요: calibration_shifted (BadZipFile); 원본 보존
train: 기존 파일 검증 중
train: 검증 완료, 재사용
validation: 기존 파일 검증 중
validation: 검증 완료, 재사용
calibration_ordinary: 기존 파일 검증 중
calibration_ordinary: 검증 완료, 재사용
calibration_shifted: 생성 시작 (총 6000개)
calibration_shifted: 1000/6000
calibration_shifted: 2000/6000
calibration_shifted: 3000/6000
calibration_shifted: 4000/6000
calibration_shifted: 5000/6000
calibration_shifted: 6000/6000
calibration_shifted: 압축 저장 및 검증 중
calibration_shifted: 저장 완료
학습·검증·정상 보정 데이터 준비 완료. 최종 평가 데이터는 아직 생성하지 않았습니다.

[완료] prepare


In [ ]:
run_followup("train")

history = pd.read_csv(EXPERIMENT / "training_history.csv")
display(history.tail(10))

print("새 가중치:", EXPERIMENT / "cnn_dr.pt")


[시작] train
저장 위치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/dr_experiment_recovered
입력 파일 해시 확인 중
{'epoch': 1, 'train_bce': 0.6567460567951202, 'validation_bce': 0.5122833491964266, 'ordinary_auc': 0.8673947987606483, 'shifted_auc': 0.8568112667694616}
{'epoch': 2, 'train_bce': 0.24288792073726653, 'validation_bce': 0.07332141602196249, 'ordinary_auc': 0.9994214805438859, 'shifted_auc': 0.9948011576803516}
{'epoch': 3, 'train_bce': 0.07753311130404472, 'validation_bce': 0.0404784172922899, 'ordinary_auc': 1.0, 'shifted_auc': 0.9980622871607971}
{'epoch': 4, 'train_bce': 0.05295128926634789, 'validation_bce': 0.03484970232383133, 'ordinary_auc': 0.9999961173190864, 'shifted_auc': 0.9983261459303907}
{'epoch': 5, 'train_bce': 0.048855096615850926, 'validation_bce': 0.04172230385192148, 'ordinary_auc': 1.0, 'shifted_auc': 0.9983467598967652}
{'epoch': 6, 'train_bce': 0.050823020830750465, 'validation_bce': 0.03323267067145835, 'ordinary_auc': 0.999992234

,epoch,train_bce,validation_bce,ordinary_auc,shifted_auc
25,26,0.028622,0.034448,1.000000,0.998141
26,27,0.028812,0.020210,1.000000,0.997320
27,28,0.028252,0.027501,1.000000,0.996821
28,29,0.027190,0.024025,0.999965,0.997271
29,30,0.033203,0.020730,1.000000,0.998227
30,31,0.030492,0.024395,1.000000,0.997576
31,32,0.031195,0.026345,1.000000,0.996850
32,33,0.030770,0.039405,1.000000,0.997992
33,34,0.028783,0.032709,1.000000,0.998161
34,35,0.028978,0.048681,0.999992,0.996945


새 가중치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/dr_experiment_recovered/cnn_dr.pt


In [ ]:
run_followup("evaluate")

results = pd.read_csv(EXPERIMENT / "final_comparison.csv")
display(results[
    [
        "model", "domain",
        "recall", "recall_lo", "recall_hi",
        "far", "far_lo", "far_hi", "auc",
    ]
])

print("결과 저장:", EXPERIMENT)

[시작] evaluate
저장 위치: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/dr_experiment_recovered
입력 파일 해시 확인 중
MLP 특징 추출 500/6000
MLP 특징 추출 1000/6000
MLP 특징 추출 1500/6000
MLP 특징 추출 2000/6000
MLP 특징 추출 2500/6000
MLP 특징 추출 3000/6000
MLP 특징 추출 3500/6000
MLP 특징 추출 4000/6000
MLP 특징 추출 4500/6000
MLP 특징 추출 5000/6000
MLP 특징 추출 5500/6000
MLP 특징 추출 6000/6000
MLP 특징 추출+CPU 추론: 22.16초
MLP 특징 추출 500/6000
MLP 특징 추출 1000/6000
MLP 특징 추출 1500/6000
MLP 특징 추출 2000/6000
MLP 특징 추출 2500/6000
MLP 특징 추출 3000/6000
MLP 특징 추출 3500/6000
MLP 특징 추출 4000/6000
MLP 특징 추출 4500/6000
MLP 특징 추출 5000/6000
MLP 특징 추출 5500/6000
MLP 특징 추출 6000/6000
MLP 특징 추출+CPU 추론: 21.05초
final_ordinary: 생성 시작 (총 4000개)
final_ordinary: 1000/4000
final_ordinary: 2000/4000
final_ordinary: 3000/4000
final_ordinary: 4000/4000
final_ordinary: 압축 저장 및 검증 중
final_ordinary: 저장 완료
MLP 특징 추출 500/4000
MLP 특징 추출 1000/4000
MLP 특징 추출 1500/4000
MLP 특징 추출 2000/4000
MLP 특징 추출 2500/4000
MLP 특징 추출 3000/4000
MLP 특징 추출 3500/4000
MLP 특징 추출 

,model,domain,recall,recall_lo,recall_hi,far,far_lo,far_hi,auc
0,v1_recalibrated,ordinary,0.9605,0.951012,0.968605,0.0000,0.000000,0.001843,0.999649
1,v1_original_threshold,ordinary,0.9990,0.996392,0.999879,0.1955,0.178325,0.213572,0.999649
2,cnn_dr,ordinary,0.9965,0.992802,0.998592,0.0000,0.000000,0.001843,0.999900
3,mlp_waveform_wall,ordinary,0.1495,0.134145,0.165890,0.0000,0.000000,0.001843,0.702245
4,v1_recalibrated,shifted,0.9640,0.954877,0.971728,0.0070,0.003832,0.011717,0.997038
5,v1_original_threshold,shifted,1.0000,0.998157,1.000000,0.9135,0.900316,0.925457,0.997038
6,cnn_dr,shifted,0.9670,0.958205,0.974388,0.0080,0.004579,0.012959,0.998077
7,mlp_waveform_wall,shifted,0.1715,0.155221,0.188748,0.0050,0.002400,0.009176,0.588177


결과 저장: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/dr_experiment_recovered


In [ ]:
import importlib.util
import json
import random
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# 기존 복구 스크립트의 모델·추론·데이터 생성 함수 재사용
module_spec = importlib.util.spec_from_file_location(
    "cnn_lowfar_helpers", SCRIPT
)
helpers = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(helpers)

SOURCE_EXPERIMENT = FOLLOWUP / "dr_experiment_recovered"

stamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
LOWFAR_RUN = FOLLOWUP / f"lowfar_005_{stamp}"
LOWFAR_RUN.mkdir(exist_ok=False)

SEED = 2026110902
TARGET_FAR = 0.005
CALIBRATION_TARGET = 0.003
MAX_EPOCHS = 45
PATIENCE = 10

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

with np.load(SOURCE_EXPERIMENT / "train.npz", allow_pickle=False) as d:
    train_rf = d["RF"]
    train_y = d["y"].astype(np.float32)

with np.load(SOURCE_EXPERIMENT / "validation.npz", allow_pickle=False) as d:
    val_rf = d["RF"]
    val_y = d["y"].astype(np.int64)
    val_domain = d["domain"]

prep = {
    "rf_mean": float(train_rf.mean(dtype=np.float64)),
    "rf_std": float(train_rf.std(dtype=np.float64)),
}
assert prep["rf_std"] > 0

train_ds = TensorDataset(
    torch.from_numpy(
        ((train_rf - prep["rf_mean"]) / prep["rf_std"])[:, None, :]
    ),
    torch.from_numpy(train_y),
)
train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

def validation_operating_point(scores):
    """각 조건에서 경험적 FAR < 0.5%가 되는 공통 임계값."""
    domain_thresholds = []

    for domain in (0, 1):
        normal = np.sort(scores[(val_domain == domain) & (val_y == 0)])
        assert len(normal) > 0

        # strict '< 0.5%' 조건을 만족하는 최대 오탐 개수
        allowed = max(0, int(np.ceil(TARGET_FAR * len(normal))) - 1)
        domain_thresholds.append(float(normal[len(normal) - allowed - 1]))

    threshold = max(domain_thresholds)
    predicted = scores > threshold

    metrics = {}
    for domain, name in [(0, "ordinary"), (1, "shifted")]:
        positive = (val_domain == domain) & (val_y == 1)
        negative = (val_domain == domain) & (val_y == 0)
        assert positive.any() and negative.any()

        metrics[f"{name}_recall"] = float(predicted[positive].mean())
        metrics[f"{name}_far"] = float(predicted[negative].mean())
        assert metrics[f"{name}_far"] < TARGET_FAR

    metrics["threshold"] = threshold
    metrics["worst_recall"] = min(
        metrics["ordinary_recall"], metrics["shifted_recall"]
    )
    metrics["mean_recall"] = (
        metrics["ordinary_recall"] + metrics["shifted_recall"]
    ) / 2
    return metrics

protocol = {
    "seed": SEED,
    "target_far_strictly_below": TARGET_FAR,
    "calibration_target": CALIBRATION_TARGET,
    "selection": (
        "검증 FAR가 두 조건 모두 0.5% 미만인 공통 임계값에서 "
        "낮은 쪽 재현율 최대화; 동률이면 평균 재현율, 이후 BCE"
    ),
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "train_sha256": helpers.sha(SOURCE_EXPERIMENT / "train.npz"),
    "validation_sha256": helpers.sha(
        SOURCE_EXPERIMENT / "validation.npz"
    ),
    "helper_sha256": helpers.sha(SCRIPT),
}
helpers.write_once(LOWFAR_RUN / "protocol.json", protocol)

print("새 실험:", LOWFAR_RUN)
print("선택 기준: 일반·shifted 모두 FAR < 0.5%에서 재현율 최대화")

새 실험: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/lowfar_005_20261109_172653
선택 기준: 일반·shifted 모두 FAR < 0.5%에서 재현율 최대화


In [ ]:
model_lowfar = helpers.PitCNN().cuda()

optimizer = torch.optim.AdamW(
    model_lowfar.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)
loss_fn = nn.BCEWithLogitsLoss()

best_key = None
bad_epochs = 0
history = []
started = time.perf_counter()

BEST_LOW = LOWFAR_RUN / "cnn_lowfar.pt"
assert not BEST_LOW.exists(), "이미 학습된 모델이 있습니다."

for epoch in range(1, MAX_EPOCHS + 1):
    model_lowfar.train()
    train_loss_sum = 0.0

    for x, y in train_loader:
        x, y = x.cuda(), y.cuda()

        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model_lowfar(x), y)
        assert torch.isfinite(loss), "학습 손실이 NaN/Inf입니다."

        loss.backward()
        nn.utils.clip_grad_norm_(model_lowfar.parameters(), 5.0)
        optimizer.step()

        train_loss_sum += loss.item() * len(y)

    model_lowfar.eval()
    val_logits = helpers.logits(
        model_lowfar, val_rf, prep, device="cuda"
    )
    val_bce = float(
        np.mean(np.logaddexp(0, val_logits) - val_y * val_logits)
    )
    metrics = validation_operating_point(val_logits)

    # 가장 취약한 조건의 재현율 → 평균 재현율 → 낮은 BCE 순
    selection_key = (
        metrics["worst_recall"],
        metrics["mean_recall"],
        -val_bce,
    )
    improved = best_key is None or selection_key > best_key

    if improved:
        best_key = selection_key
        bad_epochs = 0

        checkpoint = {
            "state_dict": {
                k: v.detach().cpu().clone()
                for k, v in model_lowfar.state_dict().items()
            },
            "preprocessing": prep,
            "epoch": epoch,
            "validation_metrics": metrics,
            "validation_bce": val_bce,
            "protocol": protocol,
        }

        temporary = BEST_LOW.with_suffix(".pt.partial")
        torch.save(checkpoint, temporary)
        # 저장본이 읽히는지 확인한 뒤 최종 이름으로 변경
        torch.load(temporary, map_location="cpu", weights_only=True)
        temporary.replace(BEST_LOW)

    else:
        bad_epochs += 1

    row = {
        "epoch": epoch,
        "train_bce": train_loss_sum / len(train_ds),
        "validation_bce": val_bce,
        **metrics,
        "best": improved,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(
        LOWFAR_RUN / "training_history.csv", index=False
    )

    print(
        f"{epoch:02d}/{MAX_EPOCHS} "
        f"일반 재현율={metrics['ordinary_recall']:.2%}, "
        f"FAR={metrics['ordinary_far']:.2%} | "
        f"shifted 재현율={metrics['shifted_recall']:.2%}, "
        f"FAR={metrics['shifted_far']:.2%} "
        f"{'★ 선택' if improved else ''}",
        flush=True,
    )

    if bad_epochs >= PATIENCE:
        print("모델 선택 지표 개선이 없어 종료합니다.")
        break

selected = torch.load(
    BEST_LOW, map_location="cpu", weights_only=True
)
print("\n선택 epoch:", selected["epoch"])
print("학습 시간:", round(time.perf_counter() - started, 1), "초")
print("저장:", BEST_LOW)

01/45 일반 재현율=76.03%, FAR=0.00% | shifted 재현율=84.11%, FAR=0.40% ★ 선택
02/45 일반 재현율=96.86%, FAR=0.00% | shifted 재현율=96.74%, FAR=0.40% ★ 선택
03/45 일반 재현율=98.62%, FAR=0.00% | shifted 재현율=97.15%, FAR=0.40% ★ 선택
04/45 일반 재현율=98.04%, FAR=0.00% | shifted 재현율=94.70%, FAR=0.40% 
05/45 일반 재현율=93.71%, FAR=0.00% | shifted 재현율=89.41%, FAR=0.40% 
06/45 일반 재현율=99.21%, FAR=0.00% | shifted 재현율=97.15%, FAR=0.40% ★ 선택
07/45 일반 재현율=99.61%, FAR=0.00% | shifted 재현율=94.70%, FAR=0.40% 
08/45 일반 재현율=98.23%, FAR=0.00% | shifted 재현율=97.35%, FAR=0.40% ★ 선택
09/45 일반 재현율=99.41%, FAR=0.00% | shifted 재현율=97.35%, FAR=0.40% ★ 선택
10/45 일반 재현율=99.41%, FAR=0.00% | shifted 재현율=97.35%, FAR=0.40% ★ 선택
11/45 일반 재현율=97.45%, FAR=0.00% | shifted 재현율=97.76%, FAR=0.40% ★ 선택
12/45 일반 재현율=100.00%, FAR=0.00% | shifted 재현율=97.35%, FAR=0.40% 
13/45 일반 재현율=99.02%, FAR=0.00% | shifted 재현율=95.93%, FAR=0.40% 
14/45 일반 재현율=99.80%, FAR=0.00% | shifted 재현율=97.76%, FAR=0.40% ★ 선택
15/45 일반 재현율=100.00%, FAR=0.00% | shifted 재현율=96.74%, FAR=0.40% 
16

In [ ]:
net, selected = helpers.checkpoint(BEST_LOW, device="cuda")
calibration_records = {}

for domain in ("ordinary", "shifted"):
    path = SOURCE_EXPERIMENT / f"calibration_{domain}.npz"

    with np.load(path, allow_pickle=False) as d:
        assert not d["y"].any(), "정상 데이터만 사용해야 합니다."
        scores = helpers.logits(
            net, d["RF"], selected["preprocessing"], device="cuda"
        )

    calibration_records[domain] = {
        **helpers.select_threshold(
            scores,
            confidence=0.975,
            target=CALIBRATION_TARGET,
        ),
        "data_sha256": helpers.sha(path),
    }

LOW_THRESHOLD = max(
    record["threshold"] for record in calibration_records.values()
)

frozen = {
    "model_sha256": helpers.sha(BEST_LOW),
    "threshold_logit": LOW_THRESHOLD,
    "decision_rule": "logit > threshold",
    "evaluation_target_far_strictly_below": TARGET_FAR,
    "calibration": calibration_records,
}
helpers.write_once(LOWFAR_RUN / "frozen_threshold.json", frozen)

print("최종 공통 임계값:", LOW_THRESHOLD)
for domain, record in calibration_records.items():
    print(
        domain,
        "정상 개수:", record["normal_n"],
        "보정 FAR 상한:", f"{record['upper']:.3%}",
    )
print("모델과 임계값 고정 완료")

최종 공통 임계값: 2.4950835704803467
ordinary 정상 개수: 6000 보정 FAR 상한: 0.285%
shifted 정상 개수: 6000 보정 FAR 상한: 0.285%
모델과 임계값 고정 완료


In [ ]:
import sys
sys.path.insert(0, str(REPO))
from maek import params

assert helpers.sha(BEST_LOW) == frozen["model_sha256"]

FINAL_CSV = LOWFAR_RUN / "final_evaluation.csv"
assert not FINAL_CSV.exists(), (
    "이미 평가 결과가 있습니다. 기존 결과를 확인하세요."
)

final_spec = helpers.protocol()
final_spec["seeds"]["final_ordinary"] = 202611092101
final_spec["seeds"]["final_shifted"] = 202611092102
final_spec["counts"]["final_ordinary"] = 8000
final_spec["counts"]["final_shifted"] = 8000

helpers.write_once(
    LOWFAR_RUN / "final_generation_protocol.json", final_spec
)
cfg = params.load_config(REPO / "config/default.yaml")

rows = []

for domain in ("ordinary", "shifted"):
    data = helpers.generate(
        f"final_{domain}", LOWFAR_RUN, cfg, final_spec
    )

    scores = helpers.logits(
        net, data["RF"], selected["preprocessing"], device="cuda"
    )
    row = helpers.summary(
        "CNN_lowFAR",
        domain,
        data["y"],
        scores,
        LOW_THRESHOLD,
    )
    row["observed_far_below_0_5_percent"] = row["far"] < TARGET_FAR
    rows.append(row)

    pd.DataFrame({
        "label": data["y"],
        "logit": scores,
        "prediction": (scores > LOW_THRESHOLD).astype(int),
    }).to_csv(
        LOWFAR_RUN / f"{domain}_predictions.csv", index=False
    )

    print(
        f"\n[{domain}] "
        f"재현율={row['recall']:.2%}, "
        f"오탐률={row['far']:.2%}, "
        f"오탐={row['far_count']}/{row['far_n']}",
        flush=True,
    )

results_lowfar = pd.DataFrame(rows)
results_lowfar.to_csv(FINAL_CSV, index=False)

display(results_lowfar[[
    "domain", "recall", "recall_lo", "recall_hi",
    "far", "far_lo", "far_hi",
    "observed_far_below_0_5_percent",
]])

print("결과 저장:", LOWFAR_RUN)

final_ordinary: 생성 시작 (총 8000개)
final_ordinary: 1000/8000
final_ordinary: 2000/8000
final_ordinary: 3000/8000
final_ordinary: 4000/8000
final_ordinary: 5000/8000
final_ordinary: 6000/8000
final_ordinary: 7000/8000
final_ordinary: 8000/8000
final_ordinary: 압축 저장 및 검증 중
final_ordinary: 저장 완료

[ordinary] 재현율=98.78%, 오탐률=0.00%, 오탐=0/4000
final_shifted: 생성 시작 (총 8000개)
final_shifted: 1000/8000
final_shifted: 2000/8000
final_shifted: 3000/8000
final_shifted: 4000/8000
final_shifted: 5000/8000
final_shifted: 6000/8000
final_shifted: 7000/8000
final_shifted: 8000/8000
final_shifted: 압축 저장 및 검증 중
final_shifted: 저장 완료

[shifted] 재현율=95.47%, 오탐률=0.05%, 오탐=2/4000


,domain,recall,recall_lo,recall_hi,far,far_lo,far_hi,observed_far_below_0_5_percent
0,ordinary,0.98775,0.983837,0.990924,0.0000,0.000000,0.000922,True
1,shifted,0.95475,0.947843,0.960982,0.0005,0.000061,0.001805,True


결과 저장: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/lowfar_005_20261109_172653


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "GPU 런타임에서 실행하세요."

DR_RUN = FOLLOWUP / "dr_experiment_recovered"
COMPARE_CSV = LOWFAR_RUN / "same_data_comparison.csv"
PAIRED_CSV = LOWFAR_RUN / "paired_changes.csv"

assert not COMPARE_CSV.exists(), (
    "이미 비교를 완료했습니다. 22번 셀만 실행하세요."
)

dr_frozen = helpers.read(DR_RUN / "frozen_comparison.json")
low_frozen = helpers.read(LOWFAR_RUN / "frozen_threshold.json")

models_to_compare = [
    {
        "name": "기존 CNN + 재보정",
        "path": DRIVE_RUN / "cnn_best.pt",
        "threshold": dr_frozen["v1_recalibrated"]["threshold"],
        "sha256": dr_frozen["v1_recalibrated"]["model_sha256"],
    },
    {
        "name": "조건 변화 재학습 CNN",
        "path": DR_RUN / "cnn_dr.pt",
        "threshold": dr_frozen["cnn_dr"]["threshold"],
        "sha256": dr_frozen["cnn_dr"]["model_sha256"],
    },
    {
        "name": "저오탐 기준 CNN",
        "path": LOWFAR_RUN / "cnn_lowfar.pt",
        "threshold": low_frozen["threshold_logit"],
        "sha256": low_frozen["model_sha256"],
    },
]

# 가중치가 임계값 확정 이후 변경되지 않았는지 확인
for item in models_to_compare:
    assert helpers.sha(item["path"]) == item["sha256"], (
        f"가중치가 변경됐습니다: {item['name']}"
    )

comparison_rows = []
paired_rows = []
prediction_frames = []
data_hashes = {}

for domain in ("ordinary", "shifted"):
    data_path = LOWFAR_RUN / f"final_{domain}.npz"
    data_hashes[domain] = helpers.sha(data_path)

    with np.load(data_path, allow_pickle=False) as data:
        signals = data["RF"]
        labels = data["y"].astype(bool)

    assert signals.shape == (8000, 4096)
    assert labels.sum() == 4000
    assert np.isfinite(signals).all()

    decisions = {}
    predictions = pd.DataFrame({
        "domain": domain,
        "row": np.arange(len(labels)),
        "label": labels.astype(int),
    })

    for item in models_to_compare:
        net, checkpoint = helpers.checkpoint(
            item["path"], device="cuda"
        )

        # 초기 GPU 실행 비용을 측정에서 제외
        with torch.inference_mode():
            for _ in range(3):
                net(torch.zeros(256, 1, 4096, device="cuda"))
        torch.cuda.synchronize()

        started = time.perf_counter()
        scores = helpers.logits(
            net,
            signals,
            checkpoint["preprocessing"],
            device="cuda",
        )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started

        predicted = scores > item["threshold"]
        decisions[item["name"]] = predicted

        row = helpers.summary(
            item["name"],
            domain,
            labels,
            scores,
            item["threshold"],
        )
        row["inference_seconds"] = elapsed
        row["batch_average_ms_per_waveform"] = (
            elapsed * 1000 / len(labels)
        )
        row["device"] = torch.cuda.get_device_name(0)
        row["observed_far_below_0_5_percent"] = row["far"] < 0.005
        comparison_rows.append(row)

        predictions[item["name"] + "_logit"] = scores
        predictions[item["name"] + "_prediction"] = predicted.astype(int)

        print(
            f"[{domain}] {item['name']} | "
            f"재현율 {row['recall']:.2%} | "
            f"오탐률 {row['far']:.2%} "
            f"({row['far_count']}/{row['far_n']}) | "
            f"{elapsed:.3f}초",
            flush=True,
        )

        del net, checkpoint

    # 같은 파형에서 실제로 무엇이 달라졌는지 계산
    current = decisions["저오탐 기준 CNN"]

    for baseline_name in ("기존 CNN + 재보정", "조건 변화 재학습 CNN"):
        baseline = decisions[baseline_name]

        paired_rows.append({
            "조건": domain,
            "비교 대상": baseline_name,
            "없어진 오탐": int(
                ((~labels) & baseline & (~current)).sum()
            ),
            "새로 생긴 오탐": int(
                ((~labels) & (~baseline) & current).sum()
            ),
            "추가로 놓친 피팅": int(
                (labels & baseline & (~current)).sum()
            ),
            "새로 탐지한 피팅": int(
                (labels & (~baseline) & current).sum()
            ),
        })

    prediction_frames.append(predictions)
    del signals

comparison = pd.DataFrame(comparison_rows)
paired = pd.DataFrame(paired_rows)

# 비교 조건과 개별 예측을 함께 저장
helpers.write_once(
    LOWFAR_RUN / "same_data_comparison_manifest.json",
    {
        "models": [
            {
                "name": item["name"],
                "sha256": item["sha256"],
                "threshold": item["threshold"],
            }
            for item in models_to_compare
        ],
        "data_sha256": data_hashes,
        "device": torch.cuda.get_device_name(0),
        "timing": (
            "batch_size=256; 정규화·CPU/GPU 전송·추론 포함; "
            "파일 읽기와 워밍업 제외; 1회 측정"
        ),
    },
)

pd.concat(prediction_frames, ignore_index=True).to_csv(
    LOWFAR_RUN / "same_data_predictions.csv", index=False
)
paired.to_csv(PAIRED_CSV, index=False)
comparison.to_csv(COMPARE_CSV, index=False)

print("\n비교 완료:", COMPARE_CSV)

[ordinary] 기존 CNN + 재보정 | 재현율 95.78% | 오탐률 0.00% (0/4000) | 0.495초
[ordinary] 조건 변화 재학습 CNN | 재현율 99.62% | 오탐률 0.00% (0/4000) | 0.393초
[ordinary] 저오탐 기준 CNN | 재현율 98.78% | 오탐률 0.00% (0/4000) | 0.367초
[shifted] 기존 CNN + 재보정 | 재현율 96.30% | 오탐률 0.38% (15/4000) | 0.367초
[shifted] 조건 변화 재학습 CNN | 재현율 97.38% | 오탐률 0.53% (21/4000) | 0.367초
[shifted] 저오탐 기준 CNN | 재현율 95.47% | 오탐률 0.05% (2/4000) | 0.366초

비교 완료: /content/maek_drive_1uof8fob/MyDrive/MAEK/UT/20261109_162703/cnn_followup_v2/lowfar_005_20261109_172653/same_data_comparison.csv


In [ ]:
comparison = pd.read_csv(LOWFAR_RUN / "same_data_comparison.csv")
paired = pd.read_csv(LOWFAR_RUN / "paired_changes.csv")

table = comparison[[
    "model", "domain",
    "recall", "far", "far_count",
    "recall_lo", "recall_hi",
    "far_lo", "far_hi",
    "inference_seconds",
]].copy()

table.columns = [
    "모델", "조건",
    "재현율", "오탐률", "오탐 개수",
    "재현율 CI 하한", "재현율 CI 상한",
    "오탐률 CI 하한", "오탐률 CI 상한",
    "추론 시간(초)",
]

print("동일 데이터 최종 비교")
display(table.style.format({
    "재현율": "{:.2%}",
    "오탐률": "{:.2%}",
    "재현율 CI 하한": "{:.2%}",
    "재현율 CI 상한": "{:.2%}",
    "오탐률 CI 하한": "{:.2%}",
    "오탐률 CI 상한": "{:.2%}",
    "추론 시간(초)": "{:.3f}",
}))

print("\n저오탐 기준 CNN으로 바꿨을 때의 변화")
display(paired)

current = comparison[
    comparison["model"] == "저오탐 기준 CNN"
]

passed = (
    len(current) == 2
    and (current["far"] < 0.005).all()
)

print(
    "\n관측 오탐률 기준:",
    "두 조건 모두 0.5% 미만 충족" if passed else "미충족 조건 있음",
)
print("가중치와 임계값은 변경하지 않았습니다.")

동일 데이터 최종 비교


,모델,조건,재현율,오탐률,오탐 개수,재현율 CI 하한,재현율 CI 상한,오탐률 CI 하한,오탐률 CI 상한,추론 시간(초)
0,기존 CNN + 재보정,ordinary,95.78%,0.00%,0,95.10%,96.38%,0.00%,0.09%,0.495
1,조건 변화 재학습 CNN,ordinary,99.62%,0.00%,0,99.38%,99.79%,0.00%,0.09%,0.393
2,저오탐 기준 CNN,ordinary,98.78%,0.00%,0,98.38%,99.09%,0.00%,0.09%,0.367
3,기존 CNN + 재보정,shifted,96.30%,0.38%,15,95.67%,96.86%,0.21%,0.62%,0.367
4,조건 변화 재학습 CNN,shifted,97.38%,0.53%,21,96.83%,97.85%,0.33%,0.80%,0.367
5,저오탐 기준 CNN,shifted,95.47%,0.05%,2,94.78%,96.10%,0.01%,0.18%,0.366



저오탐 기준 CNN으로 바꿨을 때의 변화


,조건,비교 대상,없어진 오탐,새로 생긴 오탐,추가로 놓친 피팅,새로 탐지한 피팅
0,ordinary,기존 CNN + 재보정,0,0,0,120
1,ordinary,조건 변화 재학습 CNN,0,0,34,0
2,shifted,기존 CNN + 재보정,15,2,60,27
3,shifted,조건 변화 재학습 CNN,19,0,83,7



관측 오탐률 기준: 두 조건 모두 0.5% 미만 충족
가중치와 임계값은 변경하지 않았습니다.


### 전체 실험 데이터 및 노트북 구글 드라이브 백업
로컬 작업 디렉토리(`/content/ut_cnn_runs`)와 현재 진행된 학습 결과물들을 압축하여 구글 드라이브 내 실험 폴더로 저장하고, 현재 노트북 파일도 백업합니다.

In [ ]:
import shutil
import zipfile
from pathlib import Path

# 1. 로컬 작업 폴더 전체를 Zip으로 압축
local_runs_dir = Path("/content/ut_cnn_runs")
backup_zip_path = DRIVE_RUN / f"colab_ut_cnn_runs_backup_{RUN_ID}.zip"

print("로컬 작업 폴더 압축 시작...")
with zipfile.ZipFile(backup_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in local_runs_dir.rglob('*'):
        if file_path.is_file():
            # 상대 경로 유지하여 저장
            zipf.write(file_path, file_path.relative_to(local_runs_dir.parent))

print(f"[완료] 실험 데이터 압축 완료: {backup_zip_path}")

# 2. 현재 Colab 노트북 백업 저장 (Colab 세션에서 현재 파일 복제 시도)
try:
    # 현재 실행 중인 노트북을 임시로 복사하여 드라이브에 저장
    notebooks = list(Path("/content").glob("*.ipynb"))
    if notebooks:
        for nb in notebooks:
            shutil.copy2(nb, DRIVE_RUN / f"backup_{nb.name}")
            print(f"[완료] 노트북 파일 백업 완료: {DRIVE_RUN / f'backup_{nb.name}'}")
    else:
        # 파일 브라우저 등에 노트북이 안 보일 경우를 대비하여 빈 파일 생성 방지 안내
        print("현재 /content 경로에서 직접적인 .ipynb 파일명을 찾지 못했습니다. 필요 시 좌측 파일 패널에서 마우스 우클릭으로 직접 다운로드하거나 Drive 복사본 저장을 이용해 주세요.")
except Exception as e:
    print(f"노트북 저장 중 일부 오류가 발생했습니다: {e}")

print(f"\n구글 드라이브 확인 경로: {DRIVE_RUN}")

로컬 작업 폴더 압축 시작...
[완료] 실험 데이터 압축 완료: /content/maek_drive_9gj80_mv/MyDrive/MAEK/UT/20261109_190635/colab_ut_cnn_runs_backup_20261109_190635.zip
현재 /content 경로에서 직접적인 .ipynb 파일명을 찾지 못했습니다. 필요 시 좌측 파일 패널에서 마우스 우클릭으로 직접 다운로드하거나 Drive 복사본 저장을 이용해 주세요.

구글 드라이브 확인 경로: /content/maek_drive_9gj80_mv/MyDrive/MAEK/UT/20261109_190635
